## Importing Libraries and Loading the Data

In [3]:
import pandas as pd

df = pd.read_csv("D:\PROJECT\Yuva Intern\Datasets\Crop_Wise_Area_Production_Yield\crop-wise-area-production-yield.csv")
print(df.shape)
df.head()

(455359, 16)


,id,year,state_name,state_code,district_name,district_code,crop_name,crop_code,crop_type,season,area,area_unit,production,production_unit,yield,yield_unit
0,0,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Arhar/Tur,202.0,Pulses,Kharif,21400.0,Hectare,2600.0,Tonnes,0.121,Tonnes/Hectare
1,1,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Bajra,103.0,Cereals,Kharif,1400.0,Hectare,500.0,Tonnes,0.357,Tonnes/Hectare
2,2,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Castor Seed,1002.0,Oilseeds,Kharif,1000.0,Hectare,100.0,Tonnes,0.100,Tonnes/Hectare
3,3,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Cotton(Lint),1101.0,Fiber Crops,Kharif,7300.0,Hectare,9400.0,Tonnes,1.288,Tonnes/Hectare
4,4,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Dry Chillies,502.0,Spices,Kharif,3700.0,Hectare,7100.0,Tonnes,1.919,Tonnes/Hectare


## Initial Inspection for uderstanding the data

In [4]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 455359 entries, 0 to 455358
Data columns (total 16 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               455359 non-null  int64  
 1   year             455359 non-null  object 
 2   state_name       455359 non-null  object 
 3   state_code       455359 non-null  int64  
 4   district_name    455359 non-null  object 
 5   district_code    455359 non-null  int64  
 6   crop_name        455359 non-null  object 
 7   crop_code        455338 non-null  float64
 8   crop_type        455359 non-null  object 
 9   season           455359 non-null  object 
 10  area             455359 non-null  float64
 11  area_unit        455359 non-null  object 
 12  production       450350 non-null  float64
 13  production_unit  455359 non-null  object 
 14  yield            455359 non-null  float64
 15  yield_unit       455359 non-null  object 
dtypes: float64(4), int64(3), object(9)
mem

## Detecting missing values

In [5]:
print(df.isnull().sum())

id                    0
year                  0
state_name            0
state_code            0
district_name         0
district_code         0
crop_name             0
crop_code            21
crop_type             0
season                0
area                  0
area_unit             0
production         5009
production_unit       0
yield                 0
yield_unit            0
dtype: int64


## Investigate the missing production rows before fixing them

In [6]:
print("Full row duplicates:", df.duplicated().sum())
print("Key-column duplicates:", df.duplicated(subset=['year','state_name','district_name','crop_name','season']).sum())

Full row duplicates: 0
Key-column duplicates: 0


## Cleaning the missing Production 

In [7]:
df_clean = df.dropna(subset=['production'])
print("Before:", df.shape[0], "After:", df_clean.shape[0])

Before: 455359 After: 450350


## Detect outliers correctly (per-crop,not global)

In [14]:
def flag_outliers(group):
    Q1 = group['yield'].quantile(0.25)
    Q3 = group['yield'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (group['yield'] < lower) | (group['yield'] > upper)

df_clean.groupby('crop_name', group_keys=False).apply(flag_outliers)
print("Outliers flagged:", df_clean['is_outlier'].sum(), "/", len(df_clean))

Outliers flagged: 14453 / 450350


C:\Users\sarna\AppData\Local\Temp\ipykernel_21268\1925998598.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean.groupby('crop_name', group_keys=False).apply(flag_outliers)


## Detect outliers correctly (per-crop,global)

In [15]:
Q1 = df_clean['yield'].quantile(0.25)
Q3 = df_clean['yield'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

global_outliers = (df_clean['yield'] < lower) | (df_clean['yield'] > upper)
print("Global outliers flagged:", global_outliers.sum(), "/", len(df_clean))

Global outliers flagged: 65857 / 450350


## Investigate zero/negative values

In [18]:
print("production <= 0:", (df_clean['production'] <= 0).sum())
print("yield <= 0:", (df_clean['yield'] <= 0).sum())

zero_yield = df_clean[df_clean['yield'] <= 0]
print(zero_yield[['year','state_name','crop_name','area','production','yield']].head(10))

production <= 0: 3997
yield <= 0: 4029
            year    state_name    crop_name   area  production  yield
19784  1999-2000       Haryana      Sesamum  703.0         0.0    0.0
21482  1999-2000       Haryana        Maize    4.0         0.0    0.0
22189  1999-2000       Haryana    Groundnut   10.0         0.0    0.0
22580  1999-2000       Haryana         Moth   54.0         0.0    0.0
22824  1999-2000       Haryana      Sesamum    4.0         0.0    0.0
26625  1999-2000  Chhattisgarh  Castor Seed    1.0         0.0    0.0
26630  1999-2000  Chhattisgarh      Linseed    1.0         0.0    0.0
26765  1999-2000  Chhattisgarh    Safflower    1.0         0.0    0.0
26847  1999-2000  Chhattisgarh    Safflower    2.0         0.0    0.0
26899  1999-2000       Gujarat         Moth  500.0         0.0    0.0
